In [1]:
import os
import json
import re
import kuzu
from datasets import load_dataset
from tqdm import tqdm
import spacy
import scispacy
from scispacy.linking import EntityLinker
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

/Users/tituslogo/miniforge3/envs/primekg/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nlp = spacy.load("en_core_sci_sm")
nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "linker_name": "umls"})
linker = nlp.get_pipe("scispacy_linker")
print("scispaCy loaded")

/Users/tituslogo/miniforge3/envs/primekg/lib/python3.11/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]
/Users/tituslogo/miniforge3/envs/primekg/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.1.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/tituslogo/miniforge3/envs/primekg/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.1.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn

scispaCy loaded


In [3]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedder loaded")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3574.70it/s]


Embedder loaded


In [4]:
db = kuzu.Database("primekg_kuzu")
conn = kuzu.Connection(db)
print("Connected to Kuzu")

Connected to Kuzu


In [5]:
result = conn.execute("CALL SHOW_TABLES() RETURN *")
while result.has_next():
    print(result.get_next())

[2, 'RELATES_TO', 'REL', 'local(kuzu)', '']
[0, 'Entity', 'NODE', 'local(kuzu)', '']


In [6]:
def clean_fts_term(text: str) -> str:
    cleaned = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    cleaned = re.sub(r'\b(AND|OR|NOT)\b', '', cleaned, flags=re.IGNORECASE)
    return ' '.join(cleaned.split())

In [7]:
def rank_triples_by_cosine_similarity(query_text, candidate_triples, top_k=5, min_threshold=0.25):
    if not candidate_triples:
        return []

    clean_sentences = [
        t.replace("(", "").replace(")", "").replace("[", "").replace("]", "").replace("->", " ")
        for t in candidate_triples
    ]

    query_embedding = embedder.encode([query_text])
    triple_embeddings = embedder.encode(clean_sentences)
    similarities = cosine_similarity(query_embedding, triple_embeddings)[0]

    scored = list(zip(candidate_triples, similarities))
    scored.sort(key=lambda x: x[1], reverse=True)

    return [triple for triple, score in scored if score >= min_threshold][:top_k]

In [12]:
def get_primekg_triples_with_cosine(text, max_entities=4, max_candidates_per_entity=8, top_k_final=5):
    doc = nlp(text)
    raw_entities = list(set([ent.text.strip() for ent in doc.ents if len(ent.text.strip()) > 3]))

    candidate_triples = []

    for entity in raw_entities[:max_entities]:
        fts_term = clean_fts_term(entity)
        if not fts_term or len(fts_term) < 3:
            continue

        fts_query = """
            CALL QUERY_FTS_INDEX('Entity', 'entity_fts', $search_term)
            YIELD node, score
            RETURN node.node_index AS id, node.name AS name, score
            ORDER BY score DESC
            LIMIT 1;
        """
        try:
            fts_res = conn.execute(fts_query, {"search_term": fts_term})
            if fts_res.has_next():
                node_id, matched_name, score = fts_res.get_next()
                if score < 1.0:
                    continue

                rel_query = """
                    MATCH (a:Entity)-[r:RELATES_TO]->(b:Entity)
                    WHERE a.node_index = $node_id
                    RETURN a.name AS src, r.display_relation AS rel, b.name AS dst, b.type AS dst_type
                    LIMIT $limit;
                """
                rel_res = conn.execute(rel_query, {"node_id": node_id, "limit": max_candidates_per_entity})
                while rel_res.has_next():
                    src, rel, dst, dst_type = rel_res.get_next()
                    candidate_triples.append(f"({src}) -[{rel}]-> ({dst} [{dst_type}])")
        except Exception:
            continue

    candidate_triples = list(set(candidate_triples))
    selected = rank_triples_by_cosine_similarity(text, candidate_triples, top_k=top_k_final, min_threshold=0.20)

    if not selected:
        return "No relevant Knowledge Graph relations found."
    return "\n".join(selected)

# Smoke Test

In [13]:
test_result = get_primekg_triples_with_cosine("What proteins are associated with Alzheimer's disease?")
print(test_result)

(Alzheimer disease) -[associated with]-> (AMFR [gene/protein])
(Alzheimer disease) -[associated with]-> (BCHE [gene/protein])
(Alzheimer disease) -[associated with]-> (APOC1 [gene/protein])
(Alzheimer disease) -[associated with]-> (CD2AP [gene/protein])
(Alzheimer disease) -[associated with]-> (BCL2 [gene/protein])


In [11]:
text = "What proteins are associated with Alzheimer's disease?"
doc = nlp(text)
raw_entities = list(set([ent.text.strip() for ent in doc.ents if len(ent.text.strip()) > 3]))

candidate_triples = []

for entity in raw_entities[:4]:
    fts_term = clean_fts_term(entity)
    if not fts_term or len(fts_term) < 3:
        continue

    fts_query = """
        CALL QUERY_FTS_INDEX('Entity', 'entity_fts', $search_term)
        YIELD node, score
        RETURN node.node_index AS id, node.name AS name, score
        ORDER BY score DESC
        LIMIT 1;
    """
    fts_res = conn.execute(fts_query, {"search_term": fts_term})
    if fts_res.has_next():
        node_id, matched_name, score = fts_res.get_next()
        if score < 1.0:
            continue

        rel_query = """
            MATCH (a:Entity)-[r:RELATES_TO]->(b:Entity)
            WHERE a.node_index = $node_id
            RETURN a.name AS src, r.display_relation AS rel, b.name AS dst, b.type AS dst_type
            LIMIT $limit;
        """
        rel_res = conn.execute(rel_query, {"node_id": node_id, "limit": 8})
        while rel_res.has_next():
            src, rel, dst, dst_type = rel_res.get_next()
            candidate_triples.append(f"({src}) -[{rel}]-> ({dst} [{dst_type}])")

candidate_triples = list(set(candidate_triples))
print(f"Candidate triples found: {len(candidate_triples)}")
print(candidate_triples)

selected = rank_triples_by_cosine_similarity(text, candidate_triples, top_k=5, min_threshold=0.20)
print(f"\nSelected after cosine filter: {len(selected)}")
print(selected)

Candidate triples found: 24
['(Alzheimer disease) -[associated with]-> (BCL2 [gene/protein])', '(Protein C) -[synergistic interaction]-> (Methotrexate [drug])', '(associative learning) -[interacts with]-> (AGT [gene/protein])', '(Protein C) -[synergistic interaction]-> (Cisplatin [drug])', '(associative learning) -[interacts with]-> (ACTR2 [gene/protein])', '(Protein C) -[synergistic interaction]-> (Fenoprofen [drug])', '(Alzheimer disease) -[associated with]-> (ESR1 [gene/protein])', '(Alzheimer disease) -[associated with]-> (AMFR [gene/protein])', '(associative learning) -[interacts with]-> (EIF4A3 [gene/protein])', '(Alzheimer disease) -[associated with]-> (CD2AP [gene/protein])', '(Alzheimer disease) -[associated with]-> (BIN1 [gene/protein])', '(Protein C) -[synergistic interaction]-> (Cephalexin [drug])', '(associative learning) -[interacts with]-> (SRF [gene/protein])', '(associative learning) -[interacts with]-> (ELAVL4 [gene/protein])', '(Alzheimer disease) -[associated with]-

# PubMedQA 
splitting (80/10/10)

In [ ]:
SPLIT_DIR = "PubMedQA_Split"

def ensure_raw_json_splits():
    if not os.path.exists(SPLIT_DIR):
        os.makedirs(SPLIT_DIR)
        print(f"Fetching raw PubMedQA and saving splits to '{SPLIT_DIR}/'...")

        raw_dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")

        train_testvalid = raw_dataset.train_test_split(test_size=0.2, seed=42)
        test_valid = train_testvalid['test'].train_test_split(test_size=0.5, seed=42)

        json_mapping = {
            "raw_train.json": train_testvalid['train'],
            "raw_val.json": test_valid['train'],
            "raw_test.json": test_valid['test']
        }

        for file_name, ds in json_mapping.items():
            path = os.path.join(SPLIT_DIR, file_name)
            ds.to_json(path)
            print(f"Saved: {path}")
    else:
        print(f"Found existing splits in '{SPLIT_DIR}/'.")

ensure_raw_json_splits()